In [26]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import roc_auc_score, accuracy_score, confusion_matrix, classification_report

In [27]:
# Load embeddings and clinical data
tx = pd.read_csv("data/tracerX.csv")
tx.columns = tx.columns.str.strip()
cleaned_tx_by_nan = pd.read_csv("data/20221109_TRACERx421_all_patient_df_Converted.csv")

# Drop column 'Unnamed: 0' if it exists
if 'Unnamed: 0' in cleaned_tx_by_nan.columns:
    cleaned_tx_by_nan = cleaned_tx_by_nan.drop(columns=['Unnamed: 0'])
# Rename 'patient_id' to 'cruk_id' in cleaned_tx_by_nan
if 'patient_id' in cleaned_tx_by_nan.columns:
    cleaned_tx_by_nan = cleaned_tx_by_nan.rename(columns={'patient_id': 'cruk_id'})

print(f"Columns in tx: {tx.columns.tolist()}")
print(f"Size of tx: {tx.shape}")
print(f"Columns in all_patient_df: {cleaned_tx_by_nan.columns.tolist()}")
print(f"Size of all_patient_df: {cleaned_tx_by_nan.shape}")

Columns in tx: ['cruk_id', 'tumour_id_muttable_cruk', 'tumour_id_per_patient', 'age', 'sex', 'ethnicity', 'cigs_perday', 'years_smoking', 'packyears', 'smoking_status_merged', 'is.family.lung', 'ECOG_PS', 'pathologyTNM', 'pT_stage_per_patient', 'pN_stage_per_patient', 'LVI_per_patient', 'PL_per_patient', 'margin_status_per_patient', 'size_pathology_per_patient', 'Surgery_type', 'histology_lesion1', 'histology_lesion1_merged', 'lesion1_sampled', 'histology_lesion2', 'lesion2_sampled', 'histology_multi_full', 'histology_multi_full_genomically.confirmed', 'LUAD_pred_subtype', 'adjuvant_treatment_YN', 'adjuvant_treatment_given', 'num_cycle_na.added', 'CHMPlatDgName_cleaned', 'CHMOthDgName_cleaned', 'AdjRadStartTime_manual', 'AdjRadEndTime_manual', 'Recurrence_time_use', 'newPrim_time_use', 'first_dfs_any_event_rec.or.new.primary', 'first_event_during_followup', 'cens_os', 'os_time', 'cens_dfs', 'dfs_time', 'cens_dfs_any_event', 'dfs_time_any_event', 'cens_lung_event', 'lung_event_time', 'R

In [28]:
# What columns are in tx but not in cleaned_tx_by_nan?
missing_columns = set(tx.columns) - set(cleaned_tx_by_nan.columns)
if missing_columns:
    print(f"Columns in tx not in cleaned_tx_by_nan: {missing_columns}")
    print(len(missing_columns))

Columns in tx not in cleaned_tx_by_nan: {'margin_status_per_patient', 'tumour_id_muttable_cruk', 'histology_lesion1_merged', 'tumour_id_per_patient', 'PL_per_patient'}
5


In [29]:
print(f"First 5 rows of cleaned_tx_by_nan:\n{cleaned_tx_by_nan.head()}")

First 5 rows of cleaned_tx_by_nan:
   cruk_id  age  sex  ethnicity  cigs_perday  years_smoking  packyears  \
0       32   68    0          8         20.0             35     35.000   
1      107   81    1          7         44.5             49    109.025   
2      109   60    1          7         20.0             38     38.000   
3       87   65    1          7         10.0             35     17.500   
4       43   85    1          7         10.0             25     12.500   

   smoking_status_merged  is.family.lung  ECOG_PS  ...  os_time  cens_dfs  \
0                      0               1      0.0  ...     1849         0   
1                      0               0      0.0  ...     1362         1   
2                      2               0      0.0  ...     2224         1   
3                      0               0      1.0  ...     2365         1   
4                      0               0      1.0  ...      986         1   

   dfs_time  cens_dfs_any_event  dfs_time_any_event  cens

In [30]:
# Let's use the relevant clinical features from cleaned_tx_by_nan
df_features = cleaned_tx_by_nan.copy()
# Print first few rows of 'cruk_id' in both dataframes for debugging
print("First few 'cruk_id' values in clinical features DataFrame:")
print(df_features['cruk_id'].head())
print(df_features['cruk_id'].size)

# Ensure 'cruk_id' exists in both DataFrames
if 'cruk_id' not in df_features.columns:
    raise KeyError("'cruk_id' column is missing in the clinical features DataFrame (tx_keep). Please check the input file.")

First few 'cruk_id' values in clinical features DataFrame:
0     32
1    107
2    109
3     87
4     43
Name: cruk_id, dtype: int64
421


In [31]:
# From TracerX dataset, we want to create a balanced dataset based on DFS time.
threshold_dfs_time = 365*3.5 # Set a threshold for DFS time (e.g., 3.5 year)

# Drop rows with missing dfs_time
tracerx_df = df_features.dropna(subset=['dfs_time'])

# Filter datasets
events_df = tracerx_df[tracerx_df['cens_dfs'] == 1]
non_events_df = tracerx_df[(tracerx_df['dfs_time'] > threshold_dfs_time) & (tracerx_df['cens_dfs'] == 0)]

print(f"Number of events: {len(events_df)}, Number of non-events: {len(non_events_df)}")

# Combine and label
combined_events_df = pd.concat([events_df, non_events_df], ignore_index=True)
combined_events_df['shorter_dfs_balanced'] = (combined_events_df['dfs_time'] < threshold_dfs_time).astype(int)

print(combined_events_df['shorter_dfs_balanced'].value_counts())
print(combined_events_df['shorter_dfs_balanced'].value_counts(normalize=True))

Number of events: 206, Number of non-events: 169
shorter_dfs_balanced
0    190
1    185
Name: count, dtype: int64
shorter_dfs_balanced
0    0.506667
1    0.493333
Name: proportion, dtype: float64


In [39]:
from sklearn.preprocessing import StandardScaler
# Fill NaNs with mean (or other imputation strategy)
clinical_features = [c for c in combined_events_df.columns if c not in ['shorter_dfs_balanced']]
X_clinical = combined_events_df[clinical_features].values
y = combined_events_df['shorter_dfs_balanced'].values.astype(int)

X_clinical_clean = pd.DataFrame(X_clinical).fillna(pd.DataFrame(X_clinical).mean()).values
y_clean = pd.Series(y).fillna(pd.Series(y).mean()).values

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_clinical_clean)


In [40]:
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score

# Train/test split
X_train, X_test, y_train, y_test = train_test_split(X_scaled, y_clean, test_size=0.2, random_state=42)

# Fit linear regression
linreg = LinearRegression()
linreg.fit(X_train, y_train)

# Predict
y_pred = linreg.predict(X_test)

# Evaluate
mse = mean_squared_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print(f"Linear Regression on clinical features:")
print(f"  MSE: {mse:.4f}")
print(f"  R2: {r2:.4f}")

Linear Regression on clinical features:
  MSE: 0.0352
  R2: 0.8570


### Bayesian linear regression for clinical-only dataset

In [41]:
import pymc as pm

# Standardize features
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_clinical_clean)

n_samples, n_features = X_scaled.shape

with pm.Model() as linear_model:
    # Priors
    intercept = pm.Normal("intercept", mu=0, sigma=5)
    beta = pm.Normal("beta", mu=0, sigma=2, shape=n_features)
    sigma = pm.HalfNormal("sigma", sigma=1)

    # Linear model
    mu = intercept + pm.math.dot(X_scaled, beta)

    # Likelihood
    y_obs = pm.Normal("y_obs", mu=mu, sigma=sigma, observed=y_clean)

    # Sample from posterior
    trace_lin = pm.sample(2000, tune=1000, target_accept=0.95, random_seed=42, cores=4)

with linear_model:
    post_pred = pm.sample_posterior_predictive(trace_lin, var_names=["y_obs"], random_seed=42, return_inferencedata=False)

y_pred_samples = post_pred["y_obs"]  # shape (n_draws, n_samples)

# Compute mean predictions
y_pred_mean = y_pred_samples.mean(axis=0)

Initializing NUTS using jitter+adapt_diag...
Multiprocess sampling (4 chains in 4 jobs)
NUTS: [intercept, beta, sigma]


Output()

Sampling 4 chains for 1_000 tune and 2_000 draw iterations (4_000 + 8_000 draws total) took 125 seconds.
Chain 0 reached the maximum tree depth. Increase `max_treedepth`, increase `target_accept` or reparameterize.
Chain 1 reached the maximum tree depth. Increase `max_treedepth`, increase `target_accept` or reparameterize.
Chain 2 reached the maximum tree depth. Increase `max_treedepth`, increase `target_accept` or reparameterize.
Chain 3 reached the maximum tree depth. Increase `max_treedepth`, increase `target_accept` or reparameterize.
Sampling: [y_obs]


Output()

In [62]:
print("y_pred_samples shape:", y_pred_samples.shape)
print("y_pred_samples shape:", y_pred_samples.shape)
print("y_clean shape:", y_clean.shape)

y_pred_samples shape: (4, 2000, 375)
y_pred_samples shape: (4, 2000, 375)
y_clean shape: (375,)


In [81]:
from sklearn.metrics import r2_score
import numpy as np
import pandas as pd

mse_samples = ((y_pred_samples - y_clean) ** 2).mean(axis=1)

# Collapse chains and draws -> (8000, 375)
y_pred_samples_flat = y_pred_samples.reshape(-1, y_pred_samples.shape[-1])

# Compute R² for each posterior draw
r2_samples = [
    r2_score(y_clean, y_pred_samples_flat[i, :])
    for i in range(y_pred_samples_flat.shape[0])
]

r2_samples = np.array(r2_samples)

# Posterior predictive mean
y_pred_mean = y_pred_samples_flat.mean(axis=0)
r2_mean = r2_score(y_clean, y_pred_mean)

print("Bayesian Linear Regression (Clinical Features):")
metrics_summary = pd.DataFrame([
    {
        "Metric": "R²",
        "Posterior Mean": r2_mean,
        "Mean (across draws)": r2_samples.mean(),
        "95%_CI_low": np.percentile(r2_samples, 3),
        "95%_CI_high": np.percentile(r2_samples, 97),
    },
    {
        "Metric": "MSE",
        "Posterior Mean": mse_samples,
        "Mean (across draws)": mse_samples.mean(),
        "95%_CI_low": np.percentile(mse_samples, 3),
        "95%_CI_high": np.percentile(mse_samples, 97),
    }
])

print(f"  MSE: {mse_samples.mean():.4f}")
print(f"  R2: {r2_mean.mean():.4f}")
print(metrics_summary)

Bayesian Linear Regression (Clinical Features):
  MSE: 0.0496
  R2: 0.9132
  Metric                                     Posterior Mean  \
0     R²                                            0.91316   
1    MSE  [[0.27532206626103706, 0.034674351762872924, 0...   

   Mean (across draws)  95%_CI_low  95%_CI_high  
0             0.801659    0.771363     0.828891  
1             0.049576    0.025899     0.199412  
